In [3]:
import requests
import pandas as pd
import time
import os
from datetime import datetime 

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'application/json, text/plain, */*'
}

all_products = []

print("Bắt đầu thu thập dữ liệu mỹ phẩm Tiki...")

for page in range(1, 41):
    print(f"Đang cào dữ liệu trang {page}/40...")
    url = f"https://tiki.vn/api/personalish/v1/blocks/listings?limit=40&include=advertisement&aggregations=2&version=home-persionalized&category=1582&page={page}&urlKey=cham-soc-da-mat"
    
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            data = response.json()
            
            for item in data.get('data', []):
                # 1. Xử lý giá trị bán ra 
                qty_sold = item.get('quantity_sold', {})
                sold_value = qty_sold.get('value', 0) if isinstance(qty_sold, dict) else 0
                
                raw_name = str(item.get('name', 'Unknown'))
                clean_name = raw_name.replace(',', ' -')\
                                     .replace('\n', ' ')\
                                     .replace('\r', '')\
                                     .replace("'", "")\
                                     .replace('"', '')
                
                # Bổ sung lấy brand_name để lát nữa dùng GROUP BY
                brand_name = item.get('brand_name')
                if not brand_name:
                    brand_name = 'No Brand'
                    
                product_info = {
                    'id': item.get('id'),
                    'name': clean_name,
                    'brand_name': brand_name,
                    'price': item.get('price'),
                    'discount_rate': item.get('discount_rate'),
                    'rating_average': item.get('rating_average'),
                    'review_count': item.get('review_count'),
                    'quantity_sold': sold_value
                }
                all_products.append(product_info)
        else:
            print(f"Lỗi truy cập trang {page}, Mã lỗi: {response.status_code}")
            
        time.sleep(2) 
        
    except Exception as e:
        print(f"Có lỗi ở trang {page}: {e}")
        break

# Chuyển thành DataFrame
df = pd.DataFrame(all_products)

# 3. Lọc trùng lặp
df = df.drop_duplicates(subset=['id'], keep='first')
# Bỏ dòng xóa ID để giữ lại làm Khóa chính (Primary Key) cho Iceberg

# 4. Xử lý Missing Values
numeric_cols = ['price', 'discount_rate', 'rating_average', 'review_count', 'quantity_sold']
df[numeric_cols] = df[numeric_cols].fillna(0)

# Xóa các dòng lỗi không có giá
df = df[df['price'] > 0]

# 5. Tạo biến mục tiêu (Class Attribute)
df['sales_status'] = df['quantity_sold'].apply(lambda x: 'Hot' if x >= 500 else 'Slow')

# 6. LƯU FILE CSV VỚI TÊN ĐỘNG THEO NGÀY
save_dir = '/home/jovyan/work/data'
os.makedirs(save_dir, exist_ok=True) 

# Lấy ngày hiện tại (Ví dụ: 2026-05-13)
today_str = datetime.now().strftime("%Y-%m-%d")

# Gắn ngày vào tên file
file_path = f"{save_dir}/tiki_skincare_{today_str}.csv"
df.to_csv(file_path, index=False, encoding='utf-8-sig')

print("\n" + "="*50)
print(f"Hoàn thành! Đã thu thập và làm sạch được {len(df)} sản phẩm.")
print(f"File siêu sạch lưu tại: {file_path}")
print("="*50)

Bắt đầu thu thập dữ liệu mỹ phẩm Tiki...
Đang cào dữ liệu trang 1/40...
Đang cào dữ liệu trang 2/40...
Đang cào dữ liệu trang 3/40...
Đang cào dữ liệu trang 4/40...
Đang cào dữ liệu trang 5/40...
Đang cào dữ liệu trang 6/40...
Đang cào dữ liệu trang 7/40...
Đang cào dữ liệu trang 8/40...
Đang cào dữ liệu trang 9/40...
Đang cào dữ liệu trang 10/40...
Đang cào dữ liệu trang 11/40...
Đang cào dữ liệu trang 12/40...
Đang cào dữ liệu trang 13/40...
Đang cào dữ liệu trang 14/40...
Đang cào dữ liệu trang 15/40...
Đang cào dữ liệu trang 16/40...
Đang cào dữ liệu trang 17/40...
Đang cào dữ liệu trang 18/40...
Đang cào dữ liệu trang 19/40...
Đang cào dữ liệu trang 20/40...
Đang cào dữ liệu trang 21/40...
Đang cào dữ liệu trang 22/40...
Đang cào dữ liệu trang 23/40...
Đang cào dữ liệu trang 24/40...
Đang cào dữ liệu trang 25/40...
Đang cào dữ liệu trang 26/40...
Đang cào dữ liệu trang 27/40...
Đang cào dữ liệu trang 28/40...
Đang cào dữ liệu trang 29/40...
Đang cào dữ liệu trang 30/40...
Đang cào

In [2]:
import requests
import pandas as pd
import time
import os
import json
from datetime import datetime

# ==========================================
# CẤU HÌNH HỆ THỐNG
# ==========================================
BASE_DIR = "/home/jovyan/work/data"
MASTER_CAT_FILE = os.path.join(BASE_DIR, "tiki_category.json")
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
    'Accept': 'application/json, text/plain, */*',
    'Referer': 'https://tiki.vn/'
}

# ==========================================
# 1. HÀM TRÍCH XUẤT DANH MỤC LÁ TỪ MASTER DATA
# ==========================================
def get_target_leaf_categories():
    if not os.path.exists(MASTER_CAT_FILE):
        print(f"❌ LỖI: Không tìm thấy file {MASTER_CAT_FILE}. Cần có file master trước!")
        return {}

    with open(MASTER_CAT_FILE, 'r', encoding='utf-8') as f:
        master_data = json.load(f)
        
    # Chọn 3 ngành hàng mục tiêu để làm PoC
    target_names = ["Làm Đẹp - Sức Khỏe", "Nhà Sách Tiki", "Điện Thoại - Máy Tính Bảng"]
    leaf_dict = {}

    def extract_recursive(categories, parent_name=None):
        for cat in categories:
            name = cat.get('name')
            cat_id = cat.get('id')
            children = cat.get('children', [])
            
            # Xác định ngành hàng chính (Level 1)
            current_main = parent_name
            if name in target_names:
                current_main = name
                
            # SỬ DỤNG TRƯỜNG is_leaf ĐỂ NHẬN DIỆN CHÍNH XÁC NHÁNH LÁ
            if current_main and cat.get('is_leaf') == True:
                leaf_dict[str(cat_id)] = {
                    'name': name,
                    'main_category': current_main
                }
            else:
                extract_recursive(children, current_main)

    extract_recursive(master_data.get('data', []))
    return leaf_dict

# ==========================================
# 2. CỖ MÁY CÀO VÀ ĐÓNG GÓI DỮ LIỆU
# ==========================================
def run_crawler():
    print("🚀 BẮT ĐẦU PIPELINE THU THẬP DỮ LIỆU TIKI")
    
    target_leaves = get_target_leaf_categories()
    if not target_leaves:
        return

    print(f"✅ Đã tìm thấy {len(target_leaves)} danh mục lá tiêu biểu.")
    
    all_products = []
    
    # Giới hạn 20 danh mục đầu tiên để test (bạn có thể xóa [:20] để cào toàn bộ)
    for idx, (cat_id, info) in enumerate(list(target_leaves.items())[:20]):
        print(f"\n👉 [{idx+1}/{20}] Đang quét: {info['main_category']} -> {info['name']} (ID: {cat_id})")
        
        for page in range(1, 41):
            # Radar hiển thị trực tiếp tiến độ
            print(f"   ⏳ Đang lấy trang {page}...", end="\r")
            url = f"https://tiki.vn/api/personalish/v1/blocks/listings?limit=40&category={cat_id}&page={page}"
            
            try:
                # ÁO GIÁP CHỐNG TREO: Timeout 5s kết nối, 10s tải dữ liệu
                response = requests.get(url, headers=HEADERS, timeout=(5, 10))
                
                if response.status_code == 403:
                    print(f"\n   ⚠️ Bị chặn (403) tại trang {page}. Đang chuyển danh mục!       ")
                    time.sleep(2)
                    break
                    
                if response.status_code != 200:
                    print(f"\n   ⚠️ Mã lỗi {response.status_code} tại trang {page}. Chuyển!      ")
                    break

                data = response.json()
                items = data.get('data', [])
                
                if not items:
                    print(f"   ✅ Đã vét cạn nhánh này ở trang {page-1}!" + " "*20)
                    break
                    
                for item in items:
                    amplitude = item.get('visible_impression_info', {}).get('amplitude', {})
                    product_info = {
                        'id': item.get('id'),
                        'name': str(item.get('name')).replace(',', ' '),
                        'brand': item.get('brand_name', 'N/A'),
                        'price': item.get('price', 0),
                        'discount_rate': item.get('discount_rate', 0),
                        'rating': item.get('rating_average', 0),
                        'revenue_text': item.get('quantity_sold', {}).get('text', 'Đã bán 0'),
                        'main_category': info['main_category'],
                        'sub_category': amplitude.get('category_l2_name', 'Unknown'),
                        'leaf_category': info['name']
                    }
                    all_products.append(product_info)
                
                print(f"   ✔️ Xong trang {page} (Lấy {len(items)} SP)" + " "*10)
                time.sleep(1)
                
            except Exception as e:
                # Nếu Timeout hoặc rớt mạng -> Ngắt vòng lặp page, sang danh mục mới
                print(f"\n   ❌ Lỗi/Timeout ở trang {page}. Bỏ qua nhánh này!          ")
                break

    # ==========================================
    # 3. KIỂM TRA VÀ LƯU FILE
    # ==========================================
    df = pd.DataFrame(all_products)
    
    print("\n🚀 ĐANG KIỂM TRA VÀ LÀM SẠCH DỮ LIỆU...")
    
    if df.empty:
        print("🔴 THẤT BẠI: Không cào được sản phẩm nào. Kiểm tra mạng hoặc file Master.")
        return

    if 'id' in df.columns:
        df = df.drop_duplicates(subset=['id'])
        
    if 'price' in df.columns:
        df = df[df['price'] > 0]

    today = datetime.now().strftime("%Y-%m-%d")
    output_file = os.path.join(BASE_DIR, f"tiki_multi_categories_{today}.csv")
    
    os.makedirs(BASE_DIR, exist_ok=True)
    df.to_csv(output_file, index=False, encoding='utf-8-sig')
    
    print("="*60)
    print(f"🎉 HOÀN TẤT! Đã lưu {len(df)} sản phẩm vào:")
    print(f"👉 {output_file}")
    print("="*60)

if __name__ == '__main__':
    run_crawler()

🚀 BẮT ĐẦU PIPELINE THU THẬP DỮ LIỆU TIKI
